# 02 Modern detectability of reference features

Phase 2 deliverable (Q2, part). For each reference feature and each modern layer (Sentinel-2, Sentinel-1,
ALOS PALSAR, Copernicus DEM derivatives) this notebook shows a chip centred on the feature so that you can
record whether it is **visible / ambiguous / not visible**.

Prerequisites: `sbf ingest modern export --aoi REG` (human-gated), then `sbf ingest modern build-stack --aoi REG`
(same for `INE`). If the stack is absent, the cells below print a message and carry on.

Caveats: `precision = approximate` features (TA1, PO1) are only roughly located; BASE has no geometry yet.
Buried or in-tunnel targets may have no surface signature at all (SPEC section 7).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sbf.core.aoi import load_reference_features
from sbf.ingest.modern import LAYERS, read_manifest
from sbf.ingest.modern.chips import feature_chips, stretch

HALF_WIDTH_M = 1000  # chip half-width around each feature
refs = load_reference_features()
refs = refs[refs.geometry.notna() & ~refs.geometry.is_empty]
manifests = {a: read_manifest(a) for a in sorted(refs.aoi_id.unique())}
for a, m in manifests.items():
    if m is None:
        print(f"[{a}] no stack at data/processed/modern/{a}/stack.json - run `sbf ingest modern build-stack --aoi {a}` "
              "after the (human-gated) export. Skipping this AOI.")
    else:
        print(f"[{a}] {len(m['layers'])} layers, missing: {sorted(m['missing']) or 'none'}")

In [ ]:
def show_feature(row, half_width_m=HALF_WIDTH_M, ncols=5):
    chips = feature_chips(row.aoi_id, row.geometry.x, row.geometry.y, half_width_m)
    chips = {k: v for k, v in chips.items() if v is not None}
    if not chips:
        print(f"{row.feature_id}: outside raster or no layers available")
        return
    nrows = int(np.ceil(len(chips) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (name, chip) in zip(axes.ravel(), chips.items()):
        img = np.moveaxis(chip, 0, -1) if chip.shape[0] == 3 else chip[0]
        ax.imshow(stretch(img), cmap=None if chip.shape[0] == 3 else "gray")
        ax.set_title(name, fontsize=9)
    fig.suptitle(f"{row.feature_id} ({row.precision}) - {row.name}")
    plt.show()

for row in refs.itertuples():
    if manifests.get(row.aoi_id) is None:
        print(f"{row.feature_id}: skipped (no stack for {row.aoi_id})")
        continue
    show_feature(row)

## Detectability notes (fill in after looking at the chips)

Use one of `visible`, `ambiguous`, `not visible` per cell; add a short remark (what cue, which season/polarisation).

| Feature | s2_median | s2 seasons | s1_vv/vh mean | s1_vv/vh std | palsar_hh/hv | palsar multi-year | dem_slope / dem_roughness | Remarks |
|---|---|---|---|---|---|---|---|---|
| GZ1 | | | | | | | | |
| GZ2 | | | | | | | | |
| GZ3 | | | | | | | | |
| GZ4 | | | | | | | | |
| TA1 (approx.) | | | | | | | | |
| PO1 (approx.) | | | | | | | | |
| BASE | | | | | | | | not located yet |

Summary for Q2 (modern layers only): which layers show which features, and what that implies for the detectors in Phase 4.